# Stage 3 Polybot Backtest Visualization

Loads the Kotlin `polybot-backtest` outputs, summarizes contract-level results,
and visualizes the best contract in a stage2-style workflow.


In [239]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [240]:
BACKTEST_DIR = Path('../../data/polybot_backtest')

TRADES_DIR = Path('../../data/polygon_trades_processed')
SELECTED_WALLETS_PATH = Path('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

TOP_N_CONTRACTS = 10
AGGREGATE_RESAMPLE = '1h'
AGGREGATE_MAX_CONTRACTS = None  # Set int for faster iteration during exploration


## Load summaries

In [241]:
summaries_path = BACKTEST_DIR / 'summaries.json'
if not summaries_path.exists():
    raise FileNotFoundError(f'Missing summaries file: {summaries_path}')

summaries = pd.DataFrame(json.loads(summaries_path.read_text()))
if summaries.empty:
    raise ValueError('Backtest summaries are empty.')

summaries['firstTradeAt'] = pd.to_datetime(summaries['firstTradeAt'], utc=True)
summaries['lastTradeAt'] = pd.to_datetime(summaries['lastTradeAt'], utc=True)
summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(10)


,conditionId,eventCount,orderCount,fillCount,finalRealizedPnl,finalEstimatedPnl,totalImpliedPnl,finalPosition,finalBuyVolume,finalSellVolume,violationCount,firstTradeAt,lastTradeAt,endDateIso,copyWalletPnl
6034,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,93960,1414,338,287.889930,287.900387,287.901044,202.54,3213.598383,3301.421331,0,2026-04-20 19:56:22+00:00,2026-05-01 00:34:10+00:00,2026-05-01T00:34:10Z,-6608.015818
53354,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,8280,295,52,51.312735,276.725712,276.726351,1984.86,424.866206,366.059494,0,2026-05-21 03:34:10+00:00,2026-05-27 00:28:01+00:00,2026-05-27T00:28:01Z,-2779.202721
10078,0x3094a2b925483a06aa72945a1472e311e5eb6be75284...,27018,396,46,97.304711,238.302393,238.310953,9778.53,170.284675,177.829641,0,2026-05-25 17:23:23+00:00,2026-06-15 00:21:33+00:00,2026-06-15T00:21:33Z,12230.336321
38921,0xba9e2c497a753af5534c0811d578d044695fec32a563...,9350,236,78,224.679382,224.661747,224.663691,0.60,175.611105,400.126339,0,2026-06-12 23:39:55+00:00,2026-06-19 06:04:10+00:00,2026-06-19T06:04:10Z,514.184111
15656,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,56455,1356,154,195.773578,195.769515,195.771682,17.15,1149.452445,1329.218610,0,2026-05-12 15:56:53+00:00,2026-05-27 03:10:07+00:00,2026-05-27T03:10:07Z,18543.964592
45193,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,223173,2654,514,181.216101,181.218820,181.226574,0.39,4219.419850,4400.289124,0,2026-05-15 21:28:32+00:00,2026-06-18 00:28:04+00:00,2026-06-18T00:28:04Z,162136.142854
4676,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,29836,396,40,178.657180,178.688347,178.689497,0.65,679.183697,857.709955,0,2026-06-02 19:21:06+00:00,2026-06-15 00:22:12+00:00,2026-06-15T00:22:12Z,2113.444304
13665,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a27...,60189,472,120,178.066246,178.059130,178.059356,0.97,1210.774556,1388.733860,0,2026-05-21 17:38:30+00:00,2026-05-27 06:09:07+00:00,2026-05-27T06:09:07Z,-26599.076169
44321,0xd44f161e3d609e459ebc8fa50efcf79896a83e976cf8...,1756,84,12,4.197712,174.696316,174.697932,332.32,175.895384,21.594900,0,2026-04-20 20:51:00+00:00,2026-04-28 20:41:36+00:00,2026-04-28T20:41:36Z,495.992495
45160,0xd8423c8dc3174c6839b0cf667ce1a1364f8532573940...,3550,177,15,165.203324,165.208742,165.211224,0.90,188.139999,352.998897,0,2026-04-30 06:01:22+00:00,2026-05-02 00:04:26+00:00,2026-05-02T00:04:26Z,20995.849835


In [242]:
totals = {
    'contracts_total': len(summaries),
    'contracts_with_fills': (summaries['fillCount'] > 0).sum(),
    'contracts_with_orders': (summaries['orderCount'] > 0).sum(),
    'total_estimated_pnl': summaries['finalEstimatedPnl'].sum(),
    'mean_estimated_pnl': summaries['finalEstimatedPnl'].mean(),
    'median_estimated_pnl': summaries['finalEstimatedPnl'].median(),
    'min_estimated_pnl': summaries['finalEstimatedPnl'].min(),
    'max_estimated_pnl': summaries['finalEstimatedPnl'].max(),
    'positive_pnl_contracts': (summaries['finalEstimatedPnl'] > 0).sum(),
    'negative_pnl_contracts': (summaries['finalEstimatedPnl'] < 0).sum(),
    'zero_pnl_contracts': (summaries['finalEstimatedPnl'] == 0).sum(),
    'total_orders': int(summaries['orderCount'].sum()),
    'total_fills': int(summaries['fillCount'].sum()),
    'total_events': int(summaries['eventCount'].sum()),
}

pd.DataFrame([totals]).T.rename(columns={0: 'value'})


,value
contracts_total,5.342100e+04
contracts_with_fills,5.390000e+02
contracts_with_orders,1.814000e+03
total_estimated_pnl,5.823375e+03
mean_estimated_pnl,1.090091e-01
median_estimated_pnl,0.000000e+00
min_estimated_pnl,-7.826409e+01
max_estimated_pnl,2.879004e+02
positive_pnl_contracts,4.000000e+02
negative_pnl_contracts,1.280000e+02


## Per-contract stats

In [243]:
display_cols = ['conditionId', 'finalEstimatedPnl', 'finalRealizedPnl', 'orderCount', 'fillCount', 'copyWalletPnl', 'firstTradeAt', 'lastTradeAt']

stats_sorted = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).reset_index(drop=True)

# top N
stats_sorted.head(TOP_N_CONTRACTS)[display_cols]


,conditionId,finalEstimatedPnl,finalRealizedPnl,orderCount,fillCount,copyWalletPnl,firstTradeAt,lastTradeAt
0,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,287.900387,287.889930,1414,338,-6608.015818,2026-04-20 19:56:22+00:00,2026-05-01 00:34:10+00:00
1,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,276.725712,51.312735,295,52,-2779.202721,2026-05-21 03:34:10+00:00,2026-05-27 00:28:01+00:00
2,0x3094a2b925483a06aa72945a1472e311e5eb6be75284...,238.302393,97.304711,396,46,12230.336321,2026-05-25 17:23:23+00:00,2026-06-15 00:21:33+00:00
3,0xba9e2c497a753af5534c0811d578d044695fec32a563...,224.661747,224.679382,236,78,514.184111,2026-06-12 23:39:55+00:00,2026-06-19 06:04:10+00:00
4,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9...,195.769515,195.773578,1356,154,18543.964592,2026-05-12 15:56:53+00:00,2026-05-27 03:10:07+00:00
5,0xd86a816093fcd0a0e1ca440bc5ce199bd3c5a8d6139e...,181.218820,181.216101,2654,514,162136.142854,2026-05-15 21:28:32+00:00,2026-06-18 00:28:04+00:00
6,0x16608c4fdd7cb41292f6d42c1c02b43ac22593f1c198...,178.688347,178.657180,396,40,2113.444304,2026-06-02 19:21:06+00:00,2026-06-15 00:22:12+00:00
7,0x421bc1929df1429cf2cb94f80c1ce6a3ed0d1f0b7a27...,178.059130,178.066246,472,120,-26599.076169,2026-05-21 17:38:30+00:00,2026-05-27 06:09:07+00:00
8,0xd44f161e3d609e459ebc8fa50efcf79896a83e976cf8...,174.696316,4.197712,84,12,495.992495,2026-04-20 20:51:00+00:00,2026-04-28 20:41:36+00:00
9,0xd8423c8dc3174c6839b0cf667ce1a1364f8532573940...,165.208742,165.203324,177,15,20995.849835,2026-04-30 06:01:22+00:00,2026-05-02 00:04:26+00:00


## Aggregate PnL vs wallet cohort

In [244]:
selected_wallets_all = pd.read_parquet(SELECTED_WALLETS_PATH, columns=['wallet', 'wallet_group'])
if 'wallet_group' in selected_wallets_all.columns:
    watched_wallets = set(
        selected_wallets_all.loc[selected_wallets_all['wallet_group'] == 'predicting', 'wallet']
        .astype(str)
        .str.lower()
    )
    if not watched_wallets:
        watched_wallets = set(selected_wallets_all['wallet'].astype(str).str.lower())
else:
    watched_wallets = set(selected_wallets_all['wallet'].astype(str).str.lower())

wallet_parts = []
for shard in sorted(TRADES_DIR.glob('*.parquet')):
    df = pd.read_parquet(shard, columns=['wallet', 'dt', 'is_train', 'trade_pnl'])
    df['wallet'] = df['wallet'].astype(str).str.lower()
    mask = (~df['is_train']) & (df['wallet'].isin(watched_wallets))
    wallet_parts.append(df.loc[mask, ['dt', 'trade_pnl']])

if wallet_parts:
    wallet_df = pd.concat(wallet_parts, ignore_index=True)
    wallet_df['dt'] = pd.to_datetime(wallet_df['dt'], utc=True)
else:
    wallet_df = pd.DataFrame(columns=['dt', 'trade_pnl'])

wallet_hourly = (
    wallet_df.set_index('dt')['trade_pnl']
    .resample(AGGREGATE_RESAMPLE)
    .sum()
    .rename('wallet_trade_pnl')
    .to_frame()
) if not wallet_df.empty else pd.DataFrame(columns=['wallet_trade_pnl'])

contract_dirs = sorted((BACKTEST_DIR / 'contracts').glob('*'))
if AGGREGATE_MAX_CONTRACTS is not None:
    contract_dirs = contract_dirs[:AGGREGATE_MAX_CONTRACTS]

pnl_parts = []
for contract_dir in contract_dirs:
    fills_path = contract_dir / 'fills.parquet'
    if not fills_path.exists():
        continue
    fdf = pd.read_parquet(fills_path).dropna(subset=['implied_pnl'])
    if fdf.empty:
        continue
    fdf['dt'] = pd.to_datetime(fdf['dt'], utc=True)
    pnl_parts.append(fdf)

if pnl_parts:
    polybot_all = pd.concat(pnl_parts, ignore_index=True)
    polybot_hourly = (
        polybot_all.set_index('dt')['implied_pnl']
        .resample(AGGREGATE_RESAMPLE)
        .sum()
        .rename('polybot_trade_pnl')
        .to_frame()
    )
else:
    polybot_hourly = pd.DataFrame(columns=['polybot_trade_pnl'])

combined_index = wallet_hourly.index.union(polybot_hourly.index).sort_values()
agg = pd.DataFrame(index=combined_index)
agg = agg.join(wallet_hourly, how='left').join(polybot_hourly, how='left')
agg[['wallet_trade_pnl', 'polybot_trade_pnl']] = agg[['wallet_trade_pnl', 'polybot_trade_pnl']].fillna(0.0)
agg['wallet_cum_trade_pnl'] = agg['wallet_trade_pnl'].cumsum()
agg['polybot_cum_trade_pnl'] = agg['polybot_trade_pnl'].cumsum()

aggregate_totals = pd.DataFrame([
    {'series': 'Wallet cohort cumulative trade_pnl', 'value': float(agg['wallet_cum_trade_pnl'].iloc[-1]) if not agg.empty else 0.0},
    {'series': 'Polybot cumulative traded pnl', 'value': float(agg['polybot_cum_trade_pnl'].iloc[-1]) if not agg.empty else 0.0},
    {'series': 'Polybot final estimated pnl (summaries)', 'value': float(summaries['finalEstimatedPnl'].sum())},
])
aggregate_totals


/private/tmp/ipykernel_76636/3098718116.py:46: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,series,value
0,Wallet cohort cumulative trade_pnl,163027.439181
1,Polybot cumulative traded pnl,5824.170984
2,Polybot final estimated pnl (summaries),5823.375262


In [245]:
polybot_all.groupby('condition_id').agg(
    trade_count=('implied_pnl', 'size'),
    implied_pnl_sum=('implied_pnl', 'sum')
).sort_values('implied_pnl_sum', ascending=False, key=abs).iloc[200:220]

,trade_count,implied_pnl_sum
condition_id,,
0xd73f60114a0e7169a55082daef1228cb27fa50c939eea22cb0589f6bac6ce5d3,3,5.758239
0x5f676152898df5e98578b6e3c59ed03a86193db4a6431436e150ed0698e7f9eb,3,-5.640000
0x23aa3e82120d658df2a2f077a7a88207a5a54589c839b54651ce601a6b86a7e0,6,5.615259
0x03b4dfd7d8931f708f680e96a61e89f208740b75246aae8f29fb47a40c0b6c7d,19,5.530723
0xf4b371d70e4c38017d8f846202ff01a800ceea925553dbf89c17ab88d46e0025,2,5.400000
0xf526b2fbff94ae996818e76c8b54cc99ffbfa0a1a9972a48253ada65e32786f7,20,-5.375882
0xcd26bdb63b81dfcc5482c42e9372c8a46091b1de1123d2afefb83df83fe859fd,10,5.353196
0x90bfaf5467d56bc60c35dbc373aabb15b0cefe5c04df56a7b16abe581750ae20,2,5.057143
0x5e5482b1e2afa7bed0b9a0c3c8ad33a83dd6cc6d649666cdf1977be64c42c82a,24,-5.047809


In [246]:
polybot_all#[polybot_all['condition_id'] == '0x9e564dbdf5c9ebdf0860d1970592a269a0ede28570d66a9ea2c50b6f191298d3']

,condition_id,token_id,outcome,dt,source_trade_tx_hash,trigger_wallet,trigger_tx_hash,synthetic_side,synthetic_price,synthetic_quantity,order_id,order_side,order_price,order_quantity,fill_quantity,implied_pnl,order_timestamp,accepted_at,fill_timestamp
0,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,Yes,2026-05-02 18:41:50+00:00,0x9d1349721f39ee8a832bce662b5b7ed6499d31148a3d...,0x41583f2efc720b8e2682750fffb67f2806fece9f,0xbac48690e96c65cb204cb92c4f350a2b080625b254a7...,SELL,0.200000,10.000000,order_NEW_4f545cb3,BUY,0.21,952.0,10.00,1.846765,2026-05-02T18:41:12Z,2026-05-02T18:41:50Z,2026-05-02T18:41:50Z
1,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,Yes,2026-05-02 22:31:04+00:00,0x28bf79dd745eaad7f1ce11abc927ca7257acd9ffeca0...,None,None,BUY,0.218711,232.727271,order_NEW_21856d49,SELL,0.21,10.0,10.00,-1.659656,2026-05-02T22:16:42Z,2026-05-02T22:31:04Z,2026-05-02T22:31:04Z
2,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,Yes,2026-05-02 22:40:30+00:00,0xfcd6e9c93df1fb39d799837f3bceb919f8dd48b5f5a6...,0x41583f2efc720b8e2682750fffb67f2806fece9f,0x6adb2c63b41903dde914ee9c6bb9512e88458f4f70f5...,SELL,0.200000,1.250000,order_NEW_4f4cfeaa,BUY,0.21,435.0,1.25,0.230846,2026-05-02T22:34:44Z,2026-05-02T22:40:30Z,2026-05-02T22:40:30Z
3,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,Yes,2026-05-04 15:27:06+00:00,0x57daee880d66756d36d2442d7d4f0438d62f2954dc7d...,None,None,BUY,0.210000,14.010000,order_NEW_21878e68,SELL,0.21,1.0,1.00,-0.174677,2026-05-04T15:26:42Z,2026-05-04T15:26:42Z,2026-05-04T15:27:06Z
4,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,0x0038d821cf7ab3b09bf186336c316028ea001ba3f7d7...,Yes,2026-05-05 23:01:06+00:00,0xa797705eaa0187cae6d6d852f1cd27f964ea0ea37ecf...,0xa0bca9bdd8540da95060ed1fafb78aa03835d428,0x03630ce5d5d920cb64b0cda899d8bc74270e51051073...,SELL,0.200000,1000.000000,order_NEW_4f52ec99,BUY,0.20,599.0,599.00,110.621238,2026-05-05T23:01:02Z,2026-05-05T23:01:06Z,2026-05-05T23:01:06Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8413,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,Yes,2026-05-25 02:03:19+00:00,0x51ceadb3338707f6155866de3d16fb54b305e9ab3e04...,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x0eb5a8786cf30dbf9ef9f1ecfc6304a78bddbef9ac45...,SELL,0.010000,559.550000,order_NEW_e029ffee,BUY,0.15,245.0,245.00,38.966243,2026-05-25T02:02:40Z,2026-05-25T02:02:51Z,2026-05-25T02:03:19Z
8414,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,Yes,2026-05-25 02:03:26+00:00,0xb7178244f44c27633e97449fc24e708fe4cb954bd284...,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x0eb5a8786cf30dbf9ef9f1ecfc6304a78bddbef9ac45...,SELL,0.010000,1000.000000,order_NEW_e02a0015,BUY,0.15,245.0,245.00,38.966243,2026-05-25T02:03:19Z,2026-05-25T02:03:21Z,2026-05-25T02:03:26Z
8415,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,Yes,2026-05-25 02:03:43+00:00,0x2b30df2b223b751dd58c90fb2b7e4f29449373b5eb02...,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x0eb5a8786cf30dbf9ef9f1ecfc6304a78bddbef9ac45...,SELL,0.010000,582.120000,order_NEW_e02a001c,BUY,0.15,245.0,245.00,38.966243,2026-05-25T02:03:26Z,2026-05-25T02:03:26Z,2026-05-25T02:03:43Z
8416,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,0xffb8a1a5c71eacf9b8dbe4f10b846c239c74680756cb...,Yes,2026-05-25 02:04:01+00:00,0xb763f282141de728cb145821068232e228a7fda8eb42...,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x0eb5a8786cf30dbf9ef9f1ecfc6304a78bddbef9ac45...,SELL,0.050000,21.000000,order_NEW_e02a002d,BUY,0.15,245.0,21.00,2.499964,2026-05-25T02:03:43Z,2026-05-25T02:03:52Z,2026-05-25T02:04:01Z


In [247]:
polybot_all[~polybot_all['trigger_wallet'].isna() & (polybot_all['order_side'] == 'BUY')].groupby('trigger_wallet')['implied_pnl'].sum().sort_values(ascending=False, key=abs).head(20)

trigger_wallet
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62    3055.045168
0x41583f2efc720b8e2682750fffb67f2806fece9f    2038.983333
0xf658449199d0bcf544de0ff928a3b66685f3dcfe    1963.343286
0xe732156a2d84cdfb4de831d3f11a22899e49898f    1840.757119
0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe    1832.001691
0x39d0f1dca6fb7e5514858c1a337724a426764fe8    1550.463040
0x0cb10c40b0776e9ee8cef970af85724654dda76c    1491.479919
0x92d0cb81e6c891b835c8e0272e8c323095bd269e    1068.368532
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8    1015.533225
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22     939.898068
0x927ffeba73eb63ffb7e576aca7d32f78e54be7e7     789.098232
0x1c4cd350bce3cb791daf88ce2034de9b03318d85     697.560748
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f     587.297077
0x22dbd51e1a4e10fff072fa24300238c64033190f     516.867473
0xa0bca9bdd8540da95060ed1fafb78aa03835d428    -323.831284
0x6640bd87f6e4b6e8d62457448bd1b3a4711a2202     249.108045
0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f     224.275749

In [248]:
polybot_all.groupby('trigger_wallet').agg(
    trade_count=('implied_pnl', 'size'),
    implied_pnl_sum=('implied_pnl', 'sum')
).sort_values('implied_pnl_sum', ascending=False, key=abs)

,trade_count,implied_pnl_sum
trigger_wallet,,
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,240,3055.045168
0x41583f2efc720b8e2682750fffb67f2806fece9f,225,2038.983333
0xf658449199d0bcf544de0ff928a3b66685f3dcfe,111,1963.343286
0xe732156a2d84cdfb4de831d3f11a22899e49898f,254,1840.757119
0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe,321,1832.001691
0x39d0f1dca6fb7e5514858c1a337724a426764fe8,640,1550.463040
0x0cb10c40b0776e9ee8cef970af85724654dda76c,268,1491.479919
0x92d0cb81e6c891b835c8e0272e8c323095bd269e,325,1068.368532
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,252,1015.533225


In [249]:
fig_global = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=('Cumulative PnL (all watched wallets/contracts)', 'Per-bucket traded PnL')
)

fig_global.add_trace(
    go.Scatter(x=agg.index, y=agg['wallet_cum_trade_pnl'], mode='lines', name='Wallet cohort cumulative trade_pnl'),
    row=1, col=1
)
fig_global.add_trace(
    go.Scatter(x=agg.index, y=agg['polybot_cum_trade_pnl'], mode='lines', name='Polybot cumulative traded pnl'),
    row=1, col=1
)

fig_global.add_trace(
    go.Bar(x=agg.index, y=agg['wallet_trade_pnl'], name='Wallet cohort trade_pnl (bucket)'),
    row=2, col=1
)
fig_global.add_trace(
    go.Bar(x=agg.index, y=agg['polybot_trade_pnl'], name='Polybot traded pnl (bucket)'),
    row=2, col=1
)

fig_global.update_layout(
    height=850, template='plotly_white',
    title='Global aggregation: all watched wallets/contracts vs polybot traded pnl',
    barmode='group',
)
fig_global.show()


## Order-fill delay analysis

In [250]:
# Load fill data for ALL contracts — computes delays and wallet stats in one pass
all_delays = []
wallet_parts = []

contract_dirs = sorted((BACKTEST_DIR / 'contracts').glob('*'))
if AGGREGATE_MAX_CONTRACTS is not None:
    contract_dirs = contract_dirs[:AGGREGATE_MAX_CONTRACTS]

for contract_dir in contract_dirs:
    fills_path = contract_dir / 'fills.parquet'
    if not fills_path.exists():
        continue
    df = pd.read_parquet(fills_path, columns=[
        'condition_id', 'token_id', 'dt',
        'trigger_wallet', 'synthetic_side',
        'synthetic_price', 'fill_quantity',
        'implied_pnl',
        'order_timestamp', 'fill_timestamp'
    ])
    if df.empty:
        continue
    df['dt'] = pd.to_datetime(df['dt'], utc=True)
    df['order_timestamp'] = pd.to_datetime(df['order_timestamp'], utc=True)
    df['fill_timestamp'] = pd.to_datetime(df['fill_timestamp'], utc=True)
    df['delay_s'] = (df['fill_timestamp'] - df['order_timestamp']).dt.total_seconds()
    df['notional'] = df['fill_quantity'] * df['synthetic_price']
    all_delays.append(df[['condition_id', 'delay_s']])
    wallet_parts.append(df[[
        'condition_id', 'token_id', 'dt', 'trigger_wallet',
        'synthetic_side', 'synthetic_price', 'fill_quantity',
        'notional', 'implied_pnl', 'delay_s'
    ]])

all_delays_df = pd.concat(all_delays, ignore_index=True) if all_delays else pd.DataFrame(columns=['condition_id', 'delay_s'])
all_delays_df = all_delays_df[all_delays_df['delay_s'] >= 0]

wallet_all = pd.concat(wallet_parts, ignore_index=True) if wallet_parts else pd.DataFrame(
    columns=['condition_id', 'token_id', 'dt', 'trigger_wallet', 'synthetic_side', 'synthetic_price', 'fill_quantity', 'notional', 'implied_pnl', 'delay_s']
)

print(f'Fill events loaded: {len(all_delays_df):,}')
print(f'Contracts with fills: {all_delays_df["condition_id"].nunique():,}')
print(f'Unique trigger wallets: {wallet_all["trigger_wallet"].nunique()}')


/private/tmp/ipykernel_76636/2734014879.py:22: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/private/tmp/ipykernel_76636/2734014879.py:24: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



Fill events loaded: 8,418
Contracts with fills: 539
Unique trigger wallets: 33


In [251]:
# -- ALL contracts --
delay_stats = all_delays_df['delay_s'].describe(percentiles=[.25, .5, .75, .9, .95, .99])
print('Order-fill delay stats — ALL contracts with fills')
print(delay_stats.to_string())
print()

# -- TOP-N by abs PnL --
top_ids = set(summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(TOP_N_CONTRACTS)['conditionId'])
top_delays = all_delays_df[all_delays_df['condition_id'].isin(top_ids)]

if not top_delays.empty:
    print(f'Order-fill delay stats — TOP-{TOP_N_CONTRACTS} contracts by abs PnL')
    print(f'Fill events: {len(top_delays):,} | Contracts: {top_delays["condition_id"].nunique()}')
    print(top_delays['delay_s'].describe(percentiles=[.25, .5, .75, .9, .95, .99]).to_string())
else:
    print(f'No fill events found in the top-{TOP_N_CONTRACTS} contracts.')


Order-fill delay stats — ALL contracts with fills
count      8418.000000
mean        321.884771
std        3858.902959
min           1.000000
25%           5.000000
50%          21.000000
75%          96.000000
90%         257.300000
95%         468.000000
99%        4326.880000
max      206780.000000

Order-fill delay stats — TOP-10 contracts by abs PnL
Fill events: 1,369 | Contracts: 10
count    1369.000000
mean       44.359386
std       113.628811
min         1.000000
25%         3.000000
50%        10.000000
75%        39.000000
90%       117.400000
95%       200.200000
99%       457.840000
max      1888.000000


In [252]:
import numpy as np

log_delays = np.log10(all_delays_df['delay_s'].clip(lower=1))

fig_delay = go.Figure()
fig_delay.add_trace(go.Histogram(
    x=log_delays,
    nbinsx=80,
    name='All contracts'
))

median_val = all_delays_df['delay_s'].median()
median_log = np.log10(max(median_val, 1))
fig_delay.add_vline(x=median_log, line_dash='dash', line_color='red',
                    annotation_text=f'median={median_val:.0f}s', annotation_position='top right')

max_log = int(np.ceil(log_delays.max()))
tickvals = list(range(0, max_log + 1))
ticktext = ['1s', '10s', '1m', '10m', '1.7h', '11.6h', '4.8d'][:max_log + 1]

fig_delay.update_layout(
    title='Order-fill delay distribution (all contracts) — log10 scale',
    xaxis_title='Delay',
    yaxis_title='Fill events',
    template='plotly_white', height=500,
)
fig_delay.update_xaxes(tickvals=tickvals, ticktext=ticktext)
fig_delay.show()


## Wallet-level stats

In [253]:
from collections import deque

# Aggregate wallet stats
wallet_stats = wallet_all.groupby('trigger_wallet').agg(
    fill_count=('condition_id', 'count'),
    unique_contracts=('condition_id', 'nunique'),
    total_notional=('notional', 'sum'),
    avg_notional=('notional', 'mean'),
    total_implied_pnl=('implied_pnl', 'sum'),
    median_delay_s=('delay_s', 'median'),
    avg_delay_s=('delay_s', 'mean'),
).sort_values('fill_count', ascending=False)

# FIFO PnL attribution per (condition_id, token_id)
# BUY fills go into a queue; SELL fills match against the queue
# PnL = (sell_price - buy_price) * matched_qty, attributed to SELL's trigger_wallet
pnl_per_wallet = {}

for (cond_id, token_id), grp in wallet_all.groupby(['condition_id', 'token_id'], sort=False):
    grp = grp.sort_values('dt')
    buy_queue = deque()

    for _, row in grp.iterrows():
        side = row['synthetic_side']
        qty = row['fill_quantity']
        price = row['synthetic_price']
        wallet = row['trigger_wallet'] if pd.notna(row['trigger_wallet']) else None

        if side == 'BUY':
            buy_queue.append([qty, price, wallet])
        else:
            remaining = qty
            while remaining > 1e-10 and buy_queue:
                buy_qty, buy_price, _ = buy_queue[0]
                matched = min(remaining, buy_qty)
                pnl = (price - buy_price) * matched
                if wallet is not None:
                    pnl_per_wallet[wallet] = pnl_per_wallet.get(wallet, 0.0) + pnl
                if matched >= buy_qty - 1e-10:
                    buy_queue.popleft()
                else:
                    buy_queue[0][0] -= matched
                remaining -= matched

wallet_stats['attributed_pnl'] = wallet_stats.index.to_series().map(pnl_per_wallet).fillna(0.0).round(2)
wallet_stats['total_implied_pnl'] = wallet_stats['total_implied_pnl'].round(2)

# Comparison
diff = (wallet_stats['attributed_pnl'] - wallet_stats['total_implied_pnl']).abs()
print(f'Attributed PnL total: {wallet_stats["attributed_pnl"].sum():.2f}')
print(f'Implied PnL total:    {wallet_stats["total_implied_pnl"].sum():.2f}')
print(f'Mean abs diff:        {diff.mean():.4f}')
print(f'Correlation:          {wallet_stats["attributed_pnl"].corr(wallet_stats["total_implied_pnl"]):.4f}')

wallet_stats['total_notional'] = wallet_stats['total_notional'].round(2)
wallet_stats['avg_notional'] = wallet_stats['avg_notional'].round(2)
wallet_stats['median_delay_s'] = wallet_stats['median_delay_s'].round(1)
wallet_stats['avg_delay_s'] = wallet_stats['avg_delay_s'].round(1)

print('Top 15 wallets by attributed PnL:')
wallet_stats = wallet_stats.sort_values('attributed_pnl', ascending=False, key=abs)
wallet_stats.head(15)


Attributed PnL total: -1805.56
Implied PnL total:    19875.69
Mean abs diff:        684.0555
Correlation:          -0.1058
Top 15 wallets by attributed PnL:


,fill_count,unique_contracts,total_notional,avg_notional,total_implied_pnl,median_delay_s,avg_delay_s,attributed_pnl
trigger_wallet,,,,,,,,
0xa0bca9bdd8540da95060ed1fafb78aa03835d428,412,87,6056.63,14.70,-323.83,12.5,664.2,-769.90
0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,252,38,2341.46,9.29,1015.53,6.0,75.3,474.74
0x0cb10c40b0776e9ee8cef970af85724654dda76c,268,57,3501.51,13.07,1491.48,19.0,241.3,-465.05
0x39d0f1dca6fb7e5514858c1a337724a426764fe8,640,54,7077.35,11.06,1550.46,18.0,50.7,-321.46
0x8b72cb885a6bd4ea9d1da393ca231f0fa3476dbe,321,58,3224.48,10.05,1832.00,9.0,40.6,-289.08
0xcb59ea1babf3b26ea50088057e427ab0ed8f0a22,289,58,5814.76,20.12,939.90,18.0,118.9,-282.51
0x22dbd51e1a4e10fff072fa24300238c64033190f,375,78,6721.01,17.92,516.87,19.0,301.7,235.17
0x92d0cb81e6c891b835c8e0272e8c323095bd269e,325,50,3681.87,11.33,1068.37,7.0,49.1,-224.91
0x66c1a6fe836ff555ca32848646acedbbe93bfa3f,152,62,2476.55,16.29,587.30,35.0,562.0,-220.81


In [254]:
# wallet_all[~wallet_all['trigger_wallet'].isna()]['implied_pnl'].sum()

In [255]:
top15 = wallet_stats.head(15)

fig_wallet = go.Figure()
fig_wallet.add_trace(go.Bar(
    x=top15.index.str[:10] + '...',
    y=top15['fill_count'],
    marker_color='royalblue',
    text=top15['fill_count'],
    textposition='outside',
))

fig_wallet.update_layout(
    title='Top wallets by fill count',
    xaxis_title='Wallet',
    yaxis_title='Fill count',
    template='plotly_white',
    height=500,
    margin=dict(b=120),
)
fig_wallet.update_xaxes(tickangle=45)
fig_wallet.show()


In [256]:
top15_pnl = wallet_stats.head(20)

fig_pnl = go.Figure()
colors = ['green' if v >= 0 else 'red' for v in top15_pnl['attributed_pnl']]
fig_pnl.add_trace(go.Bar(
    x=top15_pnl.index.str[:10] + '...',
    y=top15_pnl['attributed_pnl'],
    marker_color=colors,
    text=top15_pnl['attributed_pnl'].round(1),
    textposition='outside',
))

fig_pnl.update_layout(
    title='Top wallets by attributed PnL (FIFO)',
    xaxis_title='Wallet',
    yaxis_title='Attributed PnL',
    template='plotly_white',
    height=500,
    margin=dict(b=120),
)
fig_pnl.update_xaxes(tickangle=45)
fig_pnl.show()


## Top contracts PnL comparison

In [257]:
top_contracts = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).head(TOP_N_CONTRACTS).reset_index(drop=True)

pnl_parts = []
for _, row in top_contracts.iterrows():
    cid = row['conditionId']
    fills_path = BACKTEST_DIR / 'contracts' / cid / 'fills.parquet'
    if not fills_path.exists():
        continue
    fdf = pd.read_parquet(fills_path, columns=['dt', 'implied_pnl']).dropna(subset=['implied_pnl'])
    if fdf.empty:
        continue
    fdf['dt'] = pd.to_datetime(fdf['dt'], utc=True)
    fdf = fdf.sort_values('dt')
    contract_pnl = fdf.groupby('dt', as_index=False)['implied_pnl'].sum()
    contract_pnl['cum_pnl'] = contract_pnl['implied_pnl'].cumsum()
    contract_pnl['condition_id'] = cid
    pnl_parts.append(contract_pnl[['dt', 'cum_pnl', 'condition_id']])

if pnl_parts:
    pnl_all = pd.concat(pnl_parts, ignore_index=True)

    fig_top = go.Figure()
    for cid, grp in pnl_all.groupby('condition_id'):
        short_id = cid[:10] + '...' + cid[-6:]
        fig_top.add_trace(go.Scatter(
            x=grp['dt'], y=grp['cum_pnl'],
            mode='lines', name=short_id
        ))

    fig_top.update_layout(
        title=f'Cumulative PnL for top-{TOP_N_CONTRACTS} contracts by absolute PnL',
        xaxis_title='Date', yaxis_title='Cumulative PnL',
        template='plotly_white', height=600,
        legend=dict(font=dict(size=8)),
    )
    fig_top.show()
else:
    print('No fills.parquet data found for top contracts.')


## Detailed timeline: best contract

In [258]:
best_cid = summaries.sort_values('finalEstimatedPnl', ascending=False, key=abs).iloc[0]['conditionId']
best_dir = BACKTEST_DIR / 'contracts' / best_cid

parts = []
for row_type, file_name in [('stats', 'stats.parquet'), ('order', 'orders.parquet'), ('fill', 'fills.parquet'), ('public_trade', 'public_trades.parquet'), ('plan_fact', 'plan_facts.parquet')]:
    path = best_dir / file_name
    if path.exists():
        df = pd.read_parquet(path)
        df['row_type'] = row_type
        parts.append(df)

if not parts:
    raise FileNotFoundError(f'No parquet files found in {best_dir}')

events = pd.concat(parts, ignore_index=True)
events['dt'] = pd.to_datetime(events['dt'], utc=True)

print(f'Best contract: {best_cid}')
summary = json.loads((best_dir / 'summary.json').read_text())
print(f'Estimated PnL: {summary["finalEstimatedPnl"]:.2f}')
events.head()


Best contract: 0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cdc09e9d47e7d54356c6cd
Estimated PnL: 287.90


,condition_id,token_id,outcome,dt,position_after,total_buy_volume,total_sell_volume,realized_pnl,est_pnl,best_bid,...,fill_quantity,implied_pnl,order_timestamp,accepted_at,fill_timestamp,wallet,public_side,public_price,public_quantity,facts
0,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,Yes,2026-04-21 14:59:32+00:00,85.63,49.6654,0.000000,0.000000,0.801586,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,Yes,2026-04-21 14:59:48+00:00,80.71,49.6654,2.949999,0.098333,0.852075,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,Yes,2026-04-21 15:00:24+00:00,79.04,49.6654,3.949998,0.131667,0.867862,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,Yes,2026-04-21 15:02:14+00:00,75.89,49.6654,5.839998,0.194667,0.894004,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,0x1d2787cb8aed975d092b2799ed6f4083e9445f7420cd...,No,2026-04-21 15:03:08+00:00,0.00,0.0000,0.000000,0.000000,0.000000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [259]:
timeline = events[events['row_type'] == 'stats'][['dt', 'token_id', 'outcome', 'position_after', 'realized_pnl', 'est_pnl', 'best_bid', 'best_ask', 'fair_price', 'vwap5m', 'vwap15m', 'vwap1h']].copy()
timeline = timeline.sort_values(['dt', 'token_id']).reset_index(drop=True)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                     subplot_titles=('Estimated / Realized PnL', 'Position', 'Prices'))

for token_id, grp in timeline.groupby('token_id', sort=False):
    label = grp['outcome'].iloc[0]
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['est_pnl'], name=f'{label} est pnl', mode='lines'), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['realized_pnl'], name=f'{label} realized pnl', mode='lines', line=dict(dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['position_after'], name=f'{label} position', mode='lines'), row=2, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['fair_price'], name=f'{label} fair', mode='lines'), row=3, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['best_bid'], name=f'{label} bid', mode='lines', line=dict(dash='dot')), row=3, col=1)
    fig.add_trace(go.Scatter(x=grp['dt'], y=grp['best_ask'], name=f'{label} ask', mode='lines', line=dict(dash='dot')), row=3, col=1)

fig.update_layout(height=1000, template='plotly_white', title=f'Detailed timeline: {best_cid[:10]}...{best_cid[-6:]}')
fig.show()


In [260]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)

In [261]:
# Net exposition over time -- each contract resets at its resolution date
# synthetic_side is the maker's side (opposite of bot's order side)
# SELL = maker sold, bot bought (+); BUY = maker bought, bot sold (-)
side_sign = wallet_all['synthetic_side'].map({'SELL': 1, 'BUY': -1})

mkt = mdf[['condition_id', 'end_date_iso']].copy()
mkt['end_date_iso'] = pd.to_datetime(mkt['end_date_iso'], utc=True)

# Net position per contract (sum of signed fills)
contract_net_pos = (
    wallet_all.assign(signed=wallet_all['notional'] * side_sign)
    .groupby('condition_id', as_index=False)['signed'].sum()
    .rename(columns={'signed': 'net_pos'})
)

# Build reset event: at end_date_iso, subtract the contract's net position to zero it
resets = contract_net_pos.merge(mkt, on='condition_id', how='inner')
resets = resets[resets['end_date_iso'].notna()].copy()
resets = resets.rename(columns={'end_date_iso': 'dt'})
resets['signed'] = -resets['net_pos']
resets = resets[['condition_id', 'dt', 'signed']]

# Signed delta per fill: SELL synthetic = bot buys (+), BUY synthetic = bot sells (-)
fill_delta = wallet_all[['condition_id', 'dt']].assign(signed=wallet_all['notional'] * side_sign)

# Combine and cumsum per contract to get position over time
pos = pd.concat([fill_delta, resets], ignore_index=True).sort_values(['condition_id', 'dt'])
pos['exposition'] = pos.groupby('condition_id')['signed'].cumsum()
pos['exposition'] = pos['exposition'].clip(lower=0)

# Aggregate across all contracts at each timestamp
ts = pos.groupby('dt', as_index=False)['exposition'].sum().sort_values('dt')

# Cumulative PnL on secondary axis for comparison
pnl_ts = wallet_all.groupby('dt', as_index=False)['implied_pnl'].sum().sort_values('dt')
pnl_ts['cum_pnl'] = pnl_ts['implied_pnl'].cumsum()

# Clip to the fills period
fill_start = wallet_all['dt'].min()
fill_end = wallet_all['dt'].max()
ts = ts[(ts['dt'] >= fill_start) & (ts['dt'] <= fill_end)]
pnl_ts = pnl_ts[(pnl_ts['dt'] >= fill_start) & (pnl_ts['dt'] <= fill_end)]

fig_ntl = make_subplots(specs=[[{'secondary_y': True}]])

fig_ntl.add_trace(go.Scatter(
    x=ts['dt'], y=ts['exposition'],
    mode='lines', name='Net exposition (USD)',
    line=dict(color='royalblue', width=2),
), secondary_y=False)

fig_ntl.add_trace(go.Scatter(
    x=pnl_ts['dt'], y=pnl_ts['cum_pnl'],
    mode='lines', name='Cumulative implied PnL',
    line=dict(color='darkorange', width=2, dash='dot'),
), secondary_y=True)

fig_ntl.update_layout(
    title='Net exposition vs cumulative PnL over time',
    template='plotly_white', height=500,
    legend=dict(x=0.5, y=-0.2, orientation='h'),
)
fig_ntl.update_xaxes(title='Date')
fig_ntl.update_yaxes(title='Net exposition (USD)', secondary_y=False)
fig_ntl.update_yaxes(title='Cumulative implied PnL', secondary_y=True)
fig_ntl.show()
